# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata

print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Number of record sets:", len(metadata.record_sets))

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In [ ]:
# List all record sets and their details
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"RecordSet name: {rs.name} (@id={rs.id})")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    {field.name} (@id={field.id}) [type: {field.data_type}] - Description: {field.description}")
    print("-")

## 3. Data Extraction
Load data from each record set as a DataFrame for analysis. Use `@id` fields for referencing.

In [ ]:
# Prepare to load records for each record set
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns for the first record set
main_record_set_id = record_set_ids[0]  # Use the first listed record set
print(f"Columns for RecordSet {main_record_set_id}:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Remove outliers, transform data distributions, and group data by key attributes.

In [ ]:
# Example: Pick 'Age' (assuming the field uses @id 'https://sen.science/age' and exists in the record set)
# Find the numeric field to analyze
numeric_field_id = None
for field in metadata.record_sets[0].fields:
    if field.data_type in ['Float', 'Integer', 'Number'] and 'age' in field.name.lower():
        numeric_field_id = field.id
        break

if numeric_field_id is not None:
    print(f"Using numeric field: {numeric_field_id}")
    # Apply filtering, normalizing, grouping
    threshold = 60  # example threshold for age
    df = dataframes[main_record_set_id]
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Try grouping by 'Sex' (if available, find its @id)
    group_field_id = None
    for field in metadata.record_sets[0].fields:
        if 'sex' in field.name.lower():
            group_field_id = field.id
            break
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print('No numeric field (such as Age) found in the first record set.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the age distribution (if available) and compare normalized age by Sex.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If age and sex fields are present, plot histograms
if numeric_field_id is not None:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=12, kde=True)
    plt.xlabel('Age')
    plt.title('Age Distribution')
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xlabel('Sex')
        plt.ylabel('Age')
        plt.title('Age by Sex')
        plt.show()

## 6. Conclusion
Through this notebook, we loaded the FAIR^2 dataset via its Croissant schema with `mlcroissant`, overviewed its structure using `@id` references, and performed basic filtering, normalization, grouping, and visualization. These steps enable efficient exploration and analysis of clinicopathological features and stratification for colorectal cancer survivors, using well-documented, FAIR-compliant methods.